# Repaso 2: Datos semiestructurados
### Prueba de conocimientos y deducciones sobre documentos JSON sintéticos

## Propósito de la actividad

Consolidar lo estudiado sobre datos semiestructurados trabajando con documentos JSON que simulan las órdenes de una tienda en línea. La actividad repasa la lectura de JSON, el reconocimiento de su estructura (claves opcionales, anidación, listas), el tratamiento de faltantes e inconsistencias de tipo y de formato, el aplanado hacia tablas, la validación con reglas y la construcción de resúmenes anidados, para terminar con una **deducción razonada** sobre la calidad de los datos y su transformación.

## Contenidos que se repasan

- Qué son los datos semiestructurados y cómo se diferencian de las tablas (notebooks de tipos de datos y de datos no estructurados, donde la API devuelve JSON).
- Operaciones sobre datos estructurados una vez aplanados: filtrado, limpieza, conversión de tipos, agregación (notebook de datos estructurados).
- Validación y serialización de resultados.

## Instrucciones

- Trabaje las secciones en orden: varios ejercicios reutilizan resultados o funciones de ejercicios anteriores.
- Registre cada resultado en la variable `respuesta_N` (donde N es el número del ejercicio), respetando exactamente el formato pedido.
- No modifique las celdas de preparación de datos; trabaje siempre sobre copias de los datos originales.
- Los ejercicios piden **cálculos y deducciones**, no gráficos.
- Los redondeos se indican en cada enunciado. Ante un empate se aplica la regla que se indique; si no se indica ninguna, no hay empates.
- Puntaje total de la prueba: **21 puntos**.

## Preparación de los datos

Los datos son **sintéticos**: simulan las órdenes de compra de una tienda en línea que el sistema entrega como un archivo JSON. La celda siguiente genera el archivo `ordenes_tienda.json` (siempre con el mismo contenido) y lo lee como texto.

In [ ]:
import os, re, json, math, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [ ]:
from datetime import datetime, timedelta
from collections import Counter


def generar_ordenes(n=400, seed=2026):
    rng = np.random.default_rng(seed)
    ciudades = {"Quito": "Pichincha", "Guayaquil": "Guayas", "Cuenca": "Azuay", "Ambato": "Tungurahua",
                "Loja": "Loja", "Riobamba": "Chimborazo", "Latacunga": "Cotopaxi", "Píllaro": "Tungurahua"}
    lista_ciudades = list(ciudades)
    catalogo = [("SKU-101", "Teclado mecánico", "Periféricos", 45.90), ("SKU-102", "Ratón inalámbrico", "Periféricos", 18.50),
                ("SKU-103", "Monitor 24 pulgadas", "Pantallas", 139.00), ("SKU-104", "Monitor 27 pulgadas", "Pantallas", 219.00),
                ("SKU-105", "Cable HDMI", "Accesorios", 7.80), ("SKU-106", "Hub USB", "Accesorios", 16.40),
                ("SKU-107", "Disco SSD 500 GB", "Almacenamiento", 54.90), ("SKU-108", "Memoria USB 64 GB", "Almacenamiento", 9.90),
                ("SKU-109", "Audífonos con micrófono", "Audio", 29.90), ("SKU-110", "Parlante portátil", "Audio", 35.00)]
    metodos = ["tarjeta", "transferencia", "efectivo"]
    pool = ["express", "regalo", "corporativo", "primera_compra"]
    empresas = ["Servientrega", "Tramaco", "Laar"]
    inicio = datetime(2025, 3, 1)
    ordenes = []
    for i in range(1, n + 1):
        f = inicio + timedelta(minutes=int(rng.integers(0, 60 * 24 * 120)))
        orden = {"id_orden": f"ORD-{i:05d}",
                 "fecha": f.strftime("%Y-%m-%dT%H:%M:%S") if rng.random() < .70 else f.strftime("%d/%m/%Y %H:%M")}
        cid = int(rng.integers(1, 121))
        ciudad = lista_ciudades[int(rng.integers(0, len(lista_ciudades)))]
        contacto = {"email": f"cliente{cid}@correo.ec"}
        u = rng.random()
        if u < .70:
            contacto["telefono"] = "09" + "".join(str(int(x)) for x in rng.integers(0, 10, 8))
        elif u < .80:
            contacto["telefono"] = None
        elif u < .90:
            contacto["telefono"] = ""
        cliente = {"id": cid, "nombre": f"Cliente {cid:03d}", "direccion": {"ciudad": ciudad, "provincia": ciudades[ciudad]}}
        if rng.random() > .05:
            cliente["contacto"] = contacto
        orden["cliente"] = cliente
        k = int(rng.integers(1, 5))
        items = []
        for j in rng.choice(len(catalogo), size=k, replace=False):
            sku, nom, cat, precio = catalogo[int(j)]
            items.append({"sku": sku, "nombre": nom, "categoria": cat,
                          "precio": precio if rng.random() > .04 else f"{precio:.2f}",
                          "cantidad": int(rng.integers(1, 4))})
        orden["items"] = items
        metodo = str(rng.choice(metodos, p=[.55, .30, .15]))
        pago = {"metodo": metodo, "estado": str(rng.choice(["aprobado", "pendiente", "rechazado"], p=[.80, .12, .08]))}
        if metodo == "tarjeta":
            pago["cuotas"] = int(rng.choice([1, 3, 6, 12]))
        orden["pago"] = pago
        v = rng.random()
        if v < .45:
            orden["etiquetas"] = sorted(str(x) for x in rng.choice(pool, size=int(rng.integers(1, 3)), replace=False))
        elif v < .60:
            orden["etiquetas"] = []
        v = rng.random()
        if v < .75:
            orden["envio"] = {"empresa": str(rng.choice(empresas)), "costo": round(float(rng.uniform(2, 8)), 2), "dias": int(rng.integers(1, 6))}
        elif v < .85:
            orden["envio"] = None
        if rng.random() < .05:
            orden["notas"] = "cliente solicita factura"
    # (se agregan al final: ordenes con defectos de calidad)
        ordenes.append(orden)
    for pos, idx in enumerate(rng.choice(n, size=12, replace=False)):
        o = ordenes[int(idx)]
        tipo = pos % 5
        if tipo == 0:
            o["items"] = []
        elif tipo == 1:
            del o["cliente"]["id"]
        elif tipo == 2:
            del o["pago"]["metodo"]
        elif tipo == 3:
            o["items"][0]["cantidad"] = 0
        else:
            o["items"][0]["precio"] = -10.0
    return ordenes


ARCHIVO_JSON = "ordenes_tienda.json"
with open(ARCHIVO_JSON, "w", encoding="utf-8") as f:
    json.dump(generar_ordenes(), f, ensure_ascii=False, indent=1)

with open(ARCHIVO_JSON, encoding="utf-8") as f:
    texto_json = f.read()
print("Archivo:", ARCHIVO_JSON, "| caracteres:", len(texto_json))
print(texto_json[:600])

La variable `texto_json` contiene el contenido **crudo** del archivo (una cadena de texto). A partir de ella trabajará en todos los ejercicios.

# Ejercicios

## Ejercicio 1. Lectura y estructura del documento (1 punto)

**Propósito.** Comprobar que sabe convertir texto JSON en estructuras de Python y reconocer que, a diferencia de una tabla, los documentos semiestructurados no tienen todos las mismas claves.

**Enunciado.** Convierta `texto_json` en objetos de Python. Determine:

- cuántas órdenes hay,
- qué claves de primer nivel aparecen en **todas** las órdenes,
- qué claves de primer nivel aparecen solo en **algunas**.

**Formato de respuesta:** diccionario con `n_ordenes` (int), `claves_en_todas` (lista ordenada alfabéticamente) y `claves_parciales` (lista ordenada alfabéticamente).

In [ ]:
# Escriba su solución

respuesta_1 = None

## Ejercicio 2. Profundidad y rutas del documento (2 puntos)

**Propósito.** Evaluar la comprensión de la anidación: un JSON puede tener varios niveles y su estructura se describe mediante rutas de acceso.

**Enunciado.** **Profundidad** de un documento: número máximo de diccionarios anidados (el documento raíz cuenta como nivel 1; entrar a un diccionario dentro de una lista también suma un nivel).

**Ruta hoja:** camino de claves, separadas por punto, hasta un valor terminal. Los elementos de las listas **no vacías** se indican con `[]` (por ejemplo `items[].precio`). Un valor simple, `null`, una lista vacía o un diccionario vacío terminan la ruta en ese punto.

Calcule la profundidad máxima entre todas las órdenes y el número de rutas hoja **distintas** que existen en el conjunto completo.

**Formato de respuesta:** diccionario con `profundidad_maxima` (int) y `rutas_hoja_distintas` (int).

In [ ]:
# Escriba su solución

respuesta_2 = None

## Ejercicio 3. Datos faltantes en documentos (2 puntos)

**Propósito.** Evaluar el manejo de campos ausentes, nulos y vacíos, que en JSON son tres situaciones distintas que una tabla suele confundir.

**Enunciado.** Cuente:

- las órdenes **sin teléfono** de contacto: cuando falta alguna clave en el camino `cliente.contacto.telefono`, el valor es `null` o es una cadena vacía (o solo espacios),
- las órdenes **sin envío**: la clave `envio` no existe o es `null`,
- las órdenes cuyo cliente **no tiene el campo `contacto`**.

**Formato de respuesta:** diccionario con `sin_telefono`, `sin_envio` y `cliente_sin_contacto` (todos int).

In [ ]:
# Escriba su solución

respuesta_3 = None

## Ejercicio 4. Tipos inconsistentes (1 punto)

**Propósito.** Evaluar la detección y corrección de inconsistencias de tipo, frecuentes cuando los datos vienen de sistemas distintos.

**Enunciado.** Considere **todos** los ítems de todas las órdenes. Algunos precios llegan como texto en lugar de número.

Determine el total de ítems, cuántos precios llegan como texto y el precio medio por ítem tras convertir todos los precios a número (2 decimales).

**Formato de respuesta:** diccionario con `n_items` (int), `precios_texto` (int) y `precio_medio` (float).

In [ ]:
# Escriba su solución

respuesta_4 = None

## Ejercicio 5. Normalización de fechas (2 puntos)

**Propósito.** Evaluar la normalización de fechas que llegan en formatos distintos, paso imprescindible antes de cualquier análisis temporal.

**Enunciado.** El campo `fecha` llega en dos formatos: ISO (`2025-03-14T10:22:00`) y latino (`14/03/2025 10:22`). Convierta todas las fechas a un mismo tipo y determine:

- cuántas órdenes usan cada formato,
- el mes (`AAAA-MM`) con más órdenes y cuántas tiene,
- la fecha más antigua y la más reciente (formato `AAAA-MM-DD HH:MM`).

**Formato de respuesta:** diccionario con `n_iso` (int), `n_latino` (int), `mes_mas_ordenes` (str), `ordenes_mes` (int), `fecha_min` (str) y `fecha_max` (str).

In [ ]:
# Escriba su solución

respuesta_5 = None

## Ejercicio 6. Aplanado de listas anidadas (2 puntos)

**Propósito.** Evaluar la conversión de una estructura anidada (lista de ítems dentro de cada orden) a una tabla con una fila por ítem, paso habitual para analizar JSON con herramientas de datos estructurados.

**Enunciado.** Construya una tabla con **una fila por ítem** (conservando `id_orden`, `sku`, `categoria`, `cantidad` y el precio convertido a número). Defina `ingreso = precio x cantidad`.

Determine el número de filas, las unidades totales vendidas, la categoría con mayor ingreso y su ingreso (2 decimales).

**Formato de respuesta:** diccionario con `filas_items` (int), `unidades` (int), `categoria_top` (str) y `ingreso_categoria_top` (float).

In [ ]:
# Escriba su solución

respuesta_6 = None

## Ejercicio 7. Combinar niveles del documento (2 puntos)

**Propósito.** Evaluar la combinación de información de niveles distintos del documento: datos del cliente (nivel superior) con datos de los ítems (lista anidada).

**Enunciado.** Calcule el ingreso total de los ítems (`precio x cantidad`) agrupado por la **ciudad del cliente** (`cliente.direccion.ciudad`).

**Formato de respuesta:** diccionario con `ciudad` (la de mayor ingreso, str), `ingreso` (2 decimales) y `ordenes` (cantidad de órdenes de esa ciudad, int).

In [ ]:
# Escriba su solución

respuesta_7 = None

## Ejercicio 8. Listas de valores (1 punto)

**Propósito.** Evaluar el análisis de campos de tipo lista (etiquetas), que no existen en una tabla clásica.

**Enunciado.** Considere solo las órdenes cuyo campo `etiquetas` **existe y tiene al menos un elemento**. Determine cuántas son y las **3 etiquetas más frecuentes** con su frecuencia, de mayor a menor (ante empate, orden alfabético).

**Formato de respuesta:** diccionario con `ordenes_con_etiquetas` (int) y `top3` (lista de pares `[etiqueta, frecuencia]`).

In [ ]:
# Escriba su solución

respuesta_8 = None

## Ejercicio 9. Validación de esquema (3 puntos)

**Propósito.** Evaluar la capacidad de definir y aplicar reglas de calidad sobre documentos, una tarea central al recibir datos semiestructurados de fuentes externas.

**Enunciado.** Una orden es **válida** si cumple **todas** estas reglas:

1. tiene `id_orden`,
2. su `cliente` tiene `id`,
3. `items` es una lista **no vacía**,
4. su `pago` tiene `metodo`,
5. todos sus ítems tienen precio (convertido a número) **mayor que 0** y cantidad **mayor o igual a 1**.

Determine cuántas órdenes son inválidas y los identificadores de las **3 primeras** (en orden alfabético).

**Formato de respuesta:** diccionario con `n_invalidas` (int) y `primeras_3` (lista de str).

In [ ]:
# Escriba su solución

respuesta_9 = None

## Ejercicio 10. Resumen anidado y serialización (2 puntos)

**Propósito.** Evaluar la construcción de un resumen con estructura anidada, que es lo que habitualmente se devuelve a otro sistema en formato JSON.

**Enunciado.** Agrupe las órdenes por `pago.metodo` (las órdenes sin método se agrupan bajo `sin_metodo`). Para cada grupo calcule: número de órdenes, ingreso de ítems (`precio x cantidad`, 2 decimales) y cuántas órdenes tienen `pago.estado` igual a `aprobado`.

**Formato de respuesta:** diccionario `{metodo: {"ordenes": int, "ingreso_items": float, "aprobadas": int}}`, que debe poder convertirse a JSON con `json.dumps`.

In [ ]:
# Escriba su solución

respuesta_10 = None

## Ejercicio 11. Deducción razonada (3 puntos)

**Propósito.** Evaluar la capacidad de interpretar los resultados: identificar problemas de calidad, justificar por qué JSON es el formato adecuado y proponer una decisión para llevar los datos a una tabla.

**Enunciado.** Con base en los resultados de los ejercicios 3, 4, 5 y 9, redacte entre 5 y 8 líneas que expliquen:

- qué **problemas de calidad** presenta este conjunto de documentos,
- por qué un formato **semiestructurado** resulta adecuado para este tipo de dato,
- qué **decisión** tomaría para llevarlo a una tabla (qué aplanaría, qué descartaría, cómo trataría los faltantes).

Incluya al menos **tres cifras** obtenidas en sus cálculos.

**Formato de respuesta:** texto redactado en la celda de respuesta.

**Escriba aquí su respuesta:**